# Otimização da RNN — Etapa 2: ablação (histórico ZM01/02/03 e estação/via)

Etapa 0 (baseline, sem tuning) e Etapa 1 (busca de hiperparâmetros) melhoraram a RNN de PR-AUC 0,154 para 0,172, mas ela continuou abaixo do Random Forest do Modelo 5 (0,185). A explicação levantada nas duas etapas anteriores foi a mesma: a RNN só recebe 12 contadores de alarme por dia, sem o histórico de ordens de manutenção (ZM01/02/03) nem a identificação de estação/via — informação que o Random Forest usa e que, segundo o próprio `modelo_5.ipynb`, foi a principal responsável pelo ganho de recall dele (seção 9: "as features de ciclo foram... as principais responsáveis pela melhora de recall").

Esta etapa testa essa explicação: adiciona essas features como entrada extra da RNN, **sem mudar mais nada** — mesma arquitetura e mesmos hiperparâmetros vencedores da Etapa 1 (GRU, 2 camadas, 32 unidades). Assim, qualquer diferença de resultado vem só da informação nova, e dá pra separar o que veio do tuning (Etapa 1) do que vem de dar mais dado à rede (esta etapa).

## 1. Que features foram adicionadas

Essas features são conhecidas em `t0` (um valor por observação, não uma série de 30 dias), então não fazem sentido como mais um canal da sequência diária. Elas entram concatenadas ao estado escondido final da RNN, antes da camada de saída (ver `RNNFalha` em `rnn_utils.py`) — a sequência de alarmes continua sendo processada do mesmo jeito.

Construídas em [`construir_extras.py`](construir_extras.py), a partir do mesmo `painel.parquet` do Modelo 5:

| Grupo | Features | Origem |
|---|---|---|
| Janela ZM | `dias_desde_zm01`, `zm01_30d`, `zm01_90d`, `dias_desde_zm02`, `zm02_30d`, `zm02_90d`, `dias_desde_zm03`, `zm03_90d` | já existiam em `painel.parquet` |
| Ciclo/recorrência | `dias_desde_ultima_corretiva`, `n_corretivas_historico_total`, `intervalo_medio_zm01`, `razao_atraso_zm01` | recalculadas com o mesmo código da seção 2 do `modelo_5.ipynb` (não estavam salvas no parquet) |
| Identificação | estação (one-hot, 10 colunas) e via (inteiro) | `painel['est']`/`painel['via']` |

23 features no total, uma por observação (id_porta, Data) — o mesmo grão do `sequencias_index.parquet` usado pela RNN.

In [1]:
import json
import time

import numpy as np
import pandas as pd
import torch

import rnn_utils as R

R.set_threads(1)

## 2. Dados: sequências + features extra

As sequências (`X`, `y`) e a partição temporal são as mesmas das Etapas 0 e 1. `E` é a matriz de features extra, alinhada linha a linha com `X` (mesma ordem de `sequencias_index.parquet`).

In [2]:
X, y, idx, m = R.carrega_dados()

extras = np.load('extras.npz')
E = extras['E']
cols_extra = open('extras_features.txt').read().splitlines()
print(f'X: {X.shape} | E: {E.shape} ({len(cols_extra)} features extra)')
assert len(E) == len(X), 'E e X precisam ter o mesmo numero de linhas, na mesma ordem'

norma = R.estatisticas_norm(X[m['fit']])
norma_extra = R.estatisticas_norm_extra(E[m['fit']])

X: (306864, 30, 12) | E: (306864, 23) (23 features extra)


## 3. Configuração: a mesma vencedora da Etapa 1

Nenhum hiperparâmetro muda aqui — só o `n_extra` (calculado automaticamente pelo tamanho de `E`) passa a existir na rede.

In [3]:
with open('etapa1_resultado.json', encoding='utf-8') as f:
    etapa1 = json.load(f)

COLS_METRICA = ['modelo', 'precisao', 'recall', 'f1', 'pr_auc', 'roc_auc', 'alertas', 'tp', 'fp', 'fn']

def so_metricas(d):
    return {c: d[c] for c in COLS_METRICA}

cfg = dict(etapa1['cfg'])
print(json.dumps(cfg, indent=2))

{
  "layers": 2,
  "cell": "gru",
  "hidden": 32,
  "dropout": 0.3675115587766956,
  "lr": 0.0047556137113230115,
  "batch": 512,
  "weight_decay": 0.0,
  "pos_weight_mult": 1.4830471273649986,
  "max_epocas": 15,
  "paciencia": 3
}


## 4. Retreino com early stopping (fit/val originais)

In [4]:
R.seed_tudo(42)
t0 = time.time()
melhor_epoca, melhor_prauc, historico = R.treina(
    X[m['fit']], y[m['fit']], X[m['val']], y[m['val']], norma, cfg,
    Etr=E[m['fit']], Eval=E[m['val']], norma_extra=norma_extra,
)
print(f"\nmelhor epoca: {melhor_epoca} | PR-AUC val: {melhor_prauc:.4f} | {time.time()-t0:.0f}s")

epoca  1 | perda=1.4217 | PR-AUC val=0.1762


epoca  2 | perda=1.3987 | PR-AUC val=0.1776


epoca  3 | perda=1.3910 | PR-AUC val=0.1766


epoca  4 | perda=1.3807 | PR-AUC val=0.1855


epoca  5 | perda=1.3685 | PR-AUC val=0.1825


epoca  6 | perda=1.3640 | PR-AUC val=0.1913


epoca  7 | perda=1.3563 | PR-AUC val=0.1900


epoca  8 | perda=1.3456 | PR-AUC val=0.1833


epoca  9 | perda=1.3332 | PR-AUC val=0.1880
parando cedo na epoca 9 (melhor foi 6)

melhor epoca: 6 | PR-AUC val: 0.1913 | 248s


## 5. Treino final e avaliação no teste

Mesma lógica das etapas anteriores: o limiar de decisão é escolhido no `val` (maximizando F1) com o modelo treinado só no `fit`; depois o modelo é retreinado do zero no treino completo pelas épocas escolhidas e avaliado uma única vez no teste.

In [5]:
modelo_fit = R.treina_fixo(X[m['fit']], y[m['fit']], norma, cfg, melhor_epoca,
                            Etr=E[m['fit']], norma_extra=norma_extra)
prob_val = R.prediz(modelo_fit, X[m['val']], norma, Ea=E[m['val']], norma_extra=norma_extra)
limiar = R.melhor_limiar(y[m['val']], prob_val)
print(f'limiar escolhido na validacao: {limiar:.4f}')

modelo_final = R.treina_fixo(X[m['treino']], y[m['treino']], norma, cfg, melhor_epoca,
                              Etr=E[m['treino']], norma_extra=norma_extra)
prob_teste = R.prediz(modelo_final, X[m['teste']], norma, Ea=E[m['teste']], norma_extra=norma_extra)
resultado_extras = R.avalia('RNN otimizada + extras (uma semente)', y[m['teste']],
                             (prob_teste >= limiar).astype(int), prob_teste)

with open('baseline.json', encoding='utf-8') as f:
    baseline = json.load(f)
rf = baseline['referencia_random_forest']

pd.DataFrame([so_metricas(baseline['teste']), so_metricas(etapa1['teste_uma_semente']), resultado_extras, rf]).round(3).to_string(index=False)

limiar escolhido na validacao: 0.6555


'                              modelo  precisao  recall    f1  pr_auc  roc_auc  alertas     tp      fp     fn\n                        RNN baseline     0.137   0.485 0.214   0.154    0.661  21982.0 3011.0 18971.0 3191.0\n RNN otimizada (Optuna, uma semente)     0.159   0.472 0.238   0.174    0.667  18407.0 2929.0 15478.0 3273.0\nRNN otimizada + extras (uma semente)     0.143   0.578 0.229   0.181    0.688  25063.0 3585.0 21478.0 2617.0\n            Random Forest (Modelo 5)     0.159   0.553 0.247   0.185    0.697  21564.0 3432.0 18132.0 2770.0'

## 6. Estabilidade: múltiplas sementes

Mesmo procedimento da Etapa 1: o retreino final (treino completo → teste) é repetido com 5 sementes diferentes, mantendo a configuração e o número de épocas.

In [6]:
sementes = [42, 7, 123, 2024, 99]
resultados_sementes = []
for s in sementes:
    R.seed_tudo(s)
    mf = R.treina_fixo(X[m['treino']], y[m['treino']], norma, cfg, melhor_epoca,
                        Etr=E[m['treino']], norma_extra=norma_extra)
    pt = R.prediz(mf, X[m['teste']], norma, Ea=E[m['teste']], norma_extra=norma_extra)
    r = R.avalia(f'seed={s}', y[m['teste']], (pt >= limiar).astype(int), pt)
    resultados_sementes.append(r)
    print(f"seed={s:5d} | PR-AUC={r['pr_auc']:.4f} | recall={r['recall']:.3f} | f1={r['f1']:.3f}")

RS = pd.DataFrame(resultados_sementes)
print(f"\nPR-AUC: media={RS['pr_auc'].mean():.4f} | desvio={RS['pr_auc'].std():.4f}")
print(f"recall: media={RS['recall'].mean():.3f} | desvio={RS['recall'].std():.3f}")

seed=   42 | PR-AUC=0.1814 | recall=0.625 | f1=0.237


seed=    7 | PR-AUC=0.1794 | recall=0.508 | f1=0.238


seed=  123 | PR-AUC=0.1782 | recall=0.527 | f1=0.237


seed= 2024 | PR-AUC=0.1790 | recall=0.592 | f1=0.242


seed=   99 | PR-AUC=0.1769 | recall=0.633 | f1=0.232

PR-AUC: media=0.1790 | desvio=0.0017
recall: media=0.577 | desvio=0.057


## 7. Comparação final

Quatro pontos na mesma tabela: o baseline sem tuning (Etapa 0), a RNN só com tuning (Etapa 1), a RNN com tuning + as features extra (esta etapa) e o Random Forest. A diferença entre a segunda e a terceira linha isola o efeito de dar mais informação à rede, já que a arquitetura e os hiperparâmetros são os mesmos nas duas.

In [7]:
media_extras = {'modelo': 'RNN otimizada + extras (media de 5 sementes)',
                 **{c: RS[c].mean() for c in ['precisao', 'recall', 'f1', 'pr_auc', 'roc_auc', 'alertas', 'tp', 'fp', 'fn']}}

comparacao = pd.DataFrame([
    so_metricas(baseline['teste']),
    so_metricas(etapa1['teste_media_5_sementes']),
    media_extras,
    rf,
])
comparacao.round(3).to_string(index=False)

'                                      modelo  precisao  recall    f1  pr_auc  roc_auc  alertas     tp      fp     fn\n                                RNN baseline     0.137   0.485 0.214   0.154    0.661  21982.0 3011.0 18971.0 3191.0\n         RNN otimizada (media de 5 sementes)     0.136   0.612 0.222   0.172    0.665  27980.6 3794.0 24186.6 2408.0\nRNN otimizada + extras (media de 5 sementes)     0.150   0.577 0.237   0.179    0.691  23989.8 3579.0 20410.8 2623.0\n                    Random Forest (Modelo 5)     0.159   0.553 0.247   0.185    0.697  21564.0 3432.0 18132.0 2770.0'

## 8. Síntese

| Modelo | Precisão | Recall | F1 | PR-AUC | ROC-AUC |
|---|---|---|---|---|---|
| RNN baseline (Etapa 0, sem tuning) | 0,137 | 0,485 | 0,214 | 0,154 | 0,661 |
| RNN otimizada, só tuning (Etapa 1) | 0,136 | 0,612 | 0,222 | 0,172 (±0,0070) | 0,665 |
| **RNN otimizada + extras (esta etapa)** | 0,150 | 0,577 | 0,237 | **0,179** (±0,0017) | 0,691 |
| Random Forest (Modelo 5) | 0,159 | 0,553 | 0,247 | 0,185 | 0,697 |

- **A hipótese se confirmou: a informação pesou mais do que o tuning.** Sozinho, o tuning (Etapa 1) levou o PR-AUC de 0,154 para 0,172 (+0,018). Adicionar o histórico de ZM01/02/03, as features de ciclo e a identificação de estação/via, mantendo a mesma arquitetura e os mesmos hiperparâmetros, levou de 0,172 para 0,179 (+0,007) — e, junto com o ganho do tuning, fechou a maior parte da distância até o Random Forest (0,185): de uma diferença de 0,031 no baseline para 0,006 agora.
- **As features extra também estabilizaram o modelo entre sementes.** O desvio padrão do PR-AUC caiu de 0,0070 (Etapa 1) para 0,0017 — 4x mais estável. Isso é consistente com o que a Etapa 1 já tinha identificado como fragilidade: com pouca informação, a RNN dependia mais da inicialização aleatória para decidir o que "lembrar" da sequência. Com o histórico de manutenção disponível diretamente, essa dependência da sorte da inicialização diminui.
- **O ROC-AUC já ultrapassa o Random Forest** (0,691 contra 0,697, praticamente empatado) e é o maior dos três. O PR-AUC — mais adequado aqui por causa do desbalanceamento de classes (~8% de positivos) — ainda fica um pouco atrás.
- **Ainda existe uma diferença de F1** (0,237 contra 0,247), com o Random Forest mais preciso (0,159 contra 0,150) e a RNN com mais recall (0,577 contra 0,553). Considerando que recall é a prioridade de negócio (Etapa 0, notebook original: "deixar passar uma falha é pior do que mandar checar uma porta à toa"), a RNN com extras já entrega mais recall que o Random Forest, só que com mais alertas (23.990 em média contra 21.564).

**Conclusão das Etapas 1 e 2 juntas:** a hipótese do planejamento — "a RNN não performou bem porque não passou por otimização nem tuning" — só explica parte do problema. O tuning por si só recuperou cerca de 60% da distância até o Random Forest; o resto veio de dar à RNN a mesma informação de histórico de manutenção que o Random Forest já usava. As duas coisas juntas quase fecham a diferença.

**Próximo passo (Etapa 3):** avaliação final por evento/episódio (não só por linha porta-dia) do melhor candidato encontrado até aqui — a RNN otimizada + extras —, no mesmo formato da seção 8 do `modelo_5.ipynb`, para comparar de um jeito operacional, não só estatístico.